# 01 - Baseline on MATH-500
Runs the untouched base model on MATH-500 (greedy answer and majority vote over 4 samples).
Run on Kaggle (2x T4). Ran twice: `repetition_penalty=1.0` and `1.1`. Results in `experiments/00_baseline` and `experiments/08_eval_rp`.
To run with the 4B model, change `model_name`.

In [ ]:
!pip uninstall -y torchaudio # newer Kaggle image: old torchaudio breaks "import datasets" after vllm upgrades torch
!pip install -U transformers vllm math-verify datasets

In [ ]:
import sys
import torch
import transformers
import vllm
import datasets

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

dataset=load_dataset("HuggingFaceH4/MATH-500",split="test")
print(dataset)
print(dataset[0])

In [ ]:
model_name='Qwen/Qwen3.5-2B-Base' # 4B run: 'Qwen/Qwen3.5-4B-Base'
model_tag='qwen2b'                 # 4B run: 'qwen4b' (used in file names)
repetition_penalty=1.0 # 1.0 = off, 1.1 = discourage repeating loops

In [ ]:
def format_prompt(record: dict)->str:
    result=""

    result+='Question: '+record.get('problem')+'\n'

    result+='Please reason step by step, and put your final answer within \\boxed{}.'+'\n'

    result+='Answer:'

    return result

In [ ]:
prompts=[format_prompt(i) for i in dataset]
print(prompts[0])

In [ ]:
from vllm import LLM, SamplingParams

llm=LLM(
    model=model_name,
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.85,
    enforce_eager=True,
    limit_mm_per_prompt={"image": 0, "video": 0},
)

print("Model loaded")

In [ ]:
stop_words=["\n\nQuestion:", "Question:", "<|endoftext|>", "<|im_end|>"]

greedy_params=SamplingParams(
    n=1,
    temperature=0.0,
    max_tokens=2048,
    stop=stop_words,
    repetition_penalty=repetition_penalty,
)

sample_params=SamplingParams(
    n=4,
    temperature=0.7,
    top_p=0.95,
    max_tokens=2048,
    stop=stop_words,
    seed=0,
    repetition_penalty=repetition_penalty,
)

In [ ]:
greedy_outputs=llm.generate(prompts,greedy_params)
print("Greedy done")

In [ ]:
sample_outputs=llm.generate(prompts,sample_params)
print("Sampling done")

In [ ]:
def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None

    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1

    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

In [ ]:
from math_verify import parse, verify

def is_correct(text: str, gold: str)->bool:
    try:
        return bool(verify(parse('$'+gold+'$'),parse(text)))
    except Exception:
        return False

In [ ]:
from collections import Counter

results=[]

for record,greedy,sample in zip(dataset,greedy_outputs,sample_outputs):
    greedy_text=greedy.outputs[0].text
    sample_texts=[j.text for j in sample.outputs]

    votes=Counter(extract_boxed(t).strip() for t in sample_texts if extract_boxed(t))
    majority=votes.most_common(1)[0][0] if votes else None

    results.append({
        'id': record['unique_id'],
        'level': record['level'],
        'gold': record['answer'],
        'greedy_pred': extract_boxed(greedy_text),
        'greedy_ok': is_correct(greedy_text,record['answer']),
        'maj_pred': majority,
        'maj_ok': is_correct('\\boxed{'+majority+'}',record['answer']) if majority else False,
        'sample_ok': [is_correct(t,record['answer']) for t in sample_texts],
        'no_box': extract_boxed(greedy_text) is None,
        'greedy_text': greedy_text,
        'sample_texts': sample_texts,
    })

print(len(results))

In [ ]:
n=len(results)

print("pass@1 greedy:", sum(r['greedy_ok'] for r in results)/n)
print("maj@4:", sum(r['maj_ok'] for r in results)/n)
print("pass@4 (any of 4 correct):", sum(any(r['sample_ok']) for r in results)/n)
print("No boxed answer:", sum(r['no_box'] for r in results)/n)

In [ ]:
for level in sorted(set(r['level'] for r in results)):
    subset=[r for r in results if r['level']==level]
    print('Level',level,
          'n=',len(subset),
          'greedy=',round(sum(r['greedy_ok'] for r in subset)/len(subset),3),
          'maj4=',round(sum(r['maj_ok'] for r in subset)/len(subset),3))

In [ ]:
import json
from pathlib import Path

savepath=Path(f'/kaggle/working/{model_tag}_math500_baseline_rp{repetition_penalty}.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(results,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)